# 60 — Monitoring, drift & data quality

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/060-monitoring-drift-data-quality.ipynb)

A deployed model meets a moving world. Here you compute the KS statistic from two empirical CDFs, then compare it with a PSI reading and a missingness check to decide whether this is drift or a pipeline break.

## 1. KS from two CDFs

The statistic is the largest absolute gap between the reference and current CDFs. Evaluate at every observed value, not just at the reference points.

In [ ]:
import numpy as np                                    # arrays, preinstalled on Colab

ref = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90, 100])   # frozen reference
cur = np.array([10, 20, 30, 40, 50, 50, 50, 60, 70, 80])    # current window
grid = np.sort(np.concatenate([ref, cur]))             # every observed value
best_x, best_d = 0, 0.0                               # track the maximum
for x in grid:                                        # walk the support
    gap = abs((ref <= x).mean() - (cur <= x).mean())  # CDF gap at x
    if gap > best_d:                                  # keep the largest
        best_x, best_d = x, gap
print(f"max gap at x={best_x}: D={best_d:.2f}")        # expect x=50, D=0.20
print("means:", ref.mean(), cur.mean())               # 55.0 and 46.0

## 2. PSI and the quality gate

A numeric shift is only half the story. PSI checks category shares, and the missingness rates ask whether the rows themselves changed.

In [ ]:
import numpy as np                                    # arrays, preinstalled on Colab

ref_share = np.array([0.5, 0.3, 0.2])                 # reference categories
cur_share = np.array([0.4, 0.3, 0.3])                 # current categories
psi = np.sum((cur_share - ref_share) * np.log(cur_share / ref_share))   # PSI
print("PSI =", round(psi, 3))                         # 0.063, below 0.1

ref_missing, cur_missing = 0.02, 0.12                 # missingness rates
print("missing ref/current:", ref_missing, cur_missing)
if cur_missing > 2 * ref_missing:                     # simple quality gate
    print("quality gate: investigate the loader before retraining")
else:
    print("quality gate: pass, treat the shift as drift")

## Your turn

1. Change the current window to `[60, 70, 80, 90, 100, 100, 100, 100, 100, 100]`. What is D, and does the maximum gap still sit at 50?
2. Add a category share of zero to the current window. Why does PSI need a floor before taking the log?
3. The three signals here disagree — D alerts, PSI is stable, missingness doubles. Write the first action you would take and the evidence you would collect before retraining.